# Statistical Methods in Imaging (SMI) Conference, 2026.
# Empowering Large Language Models with Statistics: A Practical Tutorial for Medical Imaging
**Ernest (Khashayar) Namdar, Dominik A. Deniffel, Pascal Tyrrell**

This tutorial focuses on classifying Acute Ischemic Stroke (AIS) from Radiology reports.
Here, we enforce the LLM to output a valid JSON adhering to a Pydantic schema, containing both the binary inference and a confidence score.

Several similar pipelines were discussed in our publication:
```bibtex
@inproceedings{10.1117/12.3084682,
author = {Khashayar Namdar and Saeidehsadat Mirjalili and Lauren Erdman and Dominik A. Deniffel and Keith Brunt and Leo Anthony Celi},
title = {{Comparative evaluation of machine learning and large language model pipelines for identifying acute ischemic stroke in radiology reports}},
volume = {13926},
booktitle = {Medical Imaging 2026: Computer-Aided Diagnosis},
editor = {Axel Wism{\"u}ller and Thomas Martin Deserno},
organization = {International Society for Optics and Photonics},
publisher = {SPIE},
pages = {139261S},
keywords = {Stroke, NLP, Machine Learning, Large Language Models},
year = {2026},
doi = {10.1117/12.3084682},
URL = {https://doi.org/10.1117/12.3084682}
}
```


In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import math
import json
import re
from pydantic import BaseModel
from typing import Literal
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

torch.random.manual_seed(0)

model_name = "microsoft/MediPhi"
# Initialize the Large Language Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",          # Automatically loads the model weights onto the GPU for faster inference
    torch_dtype="auto",         # Uses the optimal precision (e.g., float16) to save GPU memory
    trust_remote_code=True,     # Required for some newer HuggingFace models with custom architectures
)
# The tokenizer converts raw text into numerical tokens the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

class StrokeInference(BaseModel):
    inference: str
    confidence: float

def build_prompt(text):
    return f"""You are a medical AI assistant trained to classify radiology MRI reports.
Determine whether there is evidence of **acute ischemic stroke** in the following report.
Output your response as a JSON object with two fields: "inference" (exactly "yes" or "no") and "confidence" (a float between 0.0 and 1.0 indicating your confidence).

Report:
---
{text}
---

Answer:"""

yes_ids = [
    tokenizer.encode("yes", add_special_tokens=False)[-1],
    tokenizer.encode("Yes", add_special_tokens=False)[-1],
    tokenizer.encode(" yes", add_special_tokens=False)[-1],
    tokenizer.encode(" Yes", add_special_tokens=False)[-1]
]
no_ids = [
    tokenizer.encode("no", add_special_tokens=False)[-1],
    tokenizer.encode("No", add_special_tokens=False)[-1],
    tokenizer.encode(" no", add_special_tokens=False)[-1],
    tokenizer.encode(" No", add_special_tokens=False)[-1]
]


In [ ]:
# Load Data
df = pd.read_csv('../data/AIS.csv')

# Chop the dataset into smaller subtasks to prevent memory bloat over thousands of iterations
num_chunks = 10
df_chunks = np.array_split(df, num_chunks)

all_probs = []
all_confidences = []
all_ids = []
all_raw_jsons = []

for i, chunk in enumerate(df_chunks):
    print(f"Processing subtask {i+1}/{num_chunks}...")
    chunk_probs = []
    chunk_confidences = []
    chunk_ids = []
    chunk_jsons = []
    
    for index, row in tqdm(chunk.iterrows(), total=len(chunk)):
        text = row['Text']
        prompt = build_prompt(text)
        messages = [
            {"role": "system", "content": "You are a radiologist."},
            {"role": "user", "content": prompt},
        ]
        
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            input_text = f"System: {messages[0]['content']}\nUser: {messages[1]['content']}\nAssistant:"
            
        # CRITICAL TRICK: Force the model to start the JSON sequence. 
        # By providing the prefix, we guarantee the very next token it generates is its "yes/no" decision!
        forced_prefix = '{\n  "inference": "'
        input_text += forced_prefix
            
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            # 1. Get Logprobs for inference
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]
        
            yes_logit = max([next_token_logits[tid].item() for tid in yes_ids])
            no_logit = max([next_token_logits[tid].item() for tid in no_ids])
            
            max_logit = max(yes_logit, no_logit)
            exp_yes = math.exp(yes_logit - max_logit)
            exp_no = math.exp(no_logit - max_logit)
            prob_yes = exp_yes / (exp_yes + exp_no)
            
            # 2. Generate the rest of the JSON
            gen_outputs = model.generate(**inputs, max_new_tokens=25, pad_token_id=tokenizer.eos_token_id)
            
        # The generated output contains the prompt + generated text.
        # Slice off the prompt to get just the generation.
        generated_sequence = tokenizer.decode(gen_outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        full_generated_json = forced_prefix + generated_sequence
        
        # Try to parse with Pydantic
        confidence_val = np.nan
        extracted_json_str = ""
        try:
            # Find JSON block just in case extra tokens generated
            match = re.search(r'\{[^{}]*\}', full_generated_json)
            if match:
                extracted_json_str = match.group(0)
                data = json.loads(extracted_json_str)
                pydantic_obj = StrokeInference(**data)
                confidence_val = pydantic_obj.confidence
        except Exception:
            pass
            
        chunk_probs.append(prob_yes)
        chunk_confidences.append(confidence_val)
        chunk_ids.append(row['ID'])
        chunk_jsons.append(full_generated_json)
        
    all_probs.extend(chunk_probs)
    all_confidences.extend(chunk_confidences)
    all_ids.extend(chunk_ids)
    all_raw_jsons.extend(chunk_jsons)
    
    temp_df = pd.DataFrame({'Id': chunk_ids, 'Prob_Yes': chunk_probs, 'Confidence': chunk_confidences, 'Raw_JSON': chunk_jsons})
    temp_df.to_csv(f'../results/LM_pydantic_inference_chunk_{i+1}.csv', index=False)
    
    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame({'Id': all_ids, 'Prob_Yes': all_probs, 'Confidence': all_confidences, 'Raw_JSON': all_raw_jsons})
results_df.to_csv('../results/LM_pydantic_inference.csv', index=False)
print("Aggregated inferences saved to ../results/LM_pydantic_inference.csv")


In [4]:
# Evaluate Probabilistic Performance Binned by Self-Reported Confidence
merged_df = results_df.merge(df[['ID', 'Label']], left_on='Id', right_on='ID')

# Filter out rows where Pydantic failed to parse a float confidence
valid_df = merged_df.dropna(subset=['Confidence']).copy()
failed_count = len(merged_df) - len(valid_df)
print(f"Rows successfully parsed: {len(valid_df)}/{len(merged_df)} (Failed: {failed_count})")

num_bins = 4 #it might not be able to dentify 4 bins if all confidence scores are the same
bins = np.linspace(0.0, 1.0, num_bins + 1)
valid_df['Conf_Bin'] = pd.cut(valid_df['Confidence'], bins=bins, include_lowest=True)

print(f"\n{'='*40}")
print(f"AUC by Confidence Bin")
print(f"{'='*40}")

for b in sorted(valid_df['Conf_Bin'].unique()):
    bin_data = valid_df[valid_df['Conf_Bin'] == b]
    n_samples = len(bin_data)
    
    if n_samples == 0:
        print(f"Bin {b}: 0 samples")
        continue
        
    y_true = bin_data['Label']
    y_prob = bin_data['Prob_Yes']
    
    if len(y_true.unique()) > 1:
        auc = roc_auc_score(y_true, y_prob)
        print(f"Bin {b}: AUC = {auc:.4f} (N={n_samples})")
    else:
        print(f"Bin {b}: Not enough class diversity to calculate AUC (N={n_samples})")


Rows successfully parsed: 3024/3024 (Failed: 0)

AUC by Confidence Bin
Bin (0.75, 1.0]: AUC = 0.9741 (N=3024)
